<a href="https://colab.research.google.com/github/sheerazshyk671/Flyrank_starter/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sheerazshyk671/Flyrank_starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I choose Lane 2: Refresh / Content Opportunity Scoring. This lane focuses on identifying which pages should be reviewed first for refresh, expansion, or monitoring. I chose this lane because it directly connects to a real operational decision — content managers have limited capacity and need to know which pages to prioritize. The starter dataset already includes signals like impressions_90d, days_since_last_update, avg_position, and trend_direction, which are exactly the types of observable signals this lane requires. Among the four predefined lanes, this one has the clearest action path: a ranked list of pages with reason codes that a reviewer can act on immediately. The starter pipeline already demonstrates that a learned model (random forest at 0.740 Precision@50) can meaningfully beat a hand-tuned baseline (0.240 Precision@50) on this type of problem, which suggests there is real signal to discover

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Search question: Which pages are most likely to benefit from a content review or refresh, based on observable search and engagement signals?

Unit of analysis: Individual pages (one row per content item)

Decision: Which pages to prioritize for content review each week

Action: Send the top-ranked pages to a content editor for review, with specific reason codes explaining why each page was flagged

Cost of a wrong call: False positives waste editor time on pages that do not actually need attention. False negatives mean declining pages go unnoticed and continue losing traffic, potentially affecting overall site performance.

Why data or ML can help: With thousands of pages in a typical content inventory, manual review of every page is impractical. Patterns of decline are often subtle and multivariate — a page might have high impressions but low CTR, or good position but stale content. A data-driven ranking can surface pages that would otherwise be missed by simple rules, while keeping the process transparent and actionable

In [2]:
import os
import sys
import subprocess

# Check if we're in Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Clone the repo if it doesn't exist
    REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
    REPO_DIR = "flyrank-ml-internship-starter"

    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        os.chdir(REPO_DIR)
    elif os.path.basename(os.getcwd()) != REPO_DIR:
        os.chdir(REPO_DIR)
else:
    # Local run: move up if running from notebooks folder
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

print(f"Current working directory: {os.getcwd()}")
print(f"Files in data/raw/: {os.listdir('data/raw') if os.path.exists('data/raw') else 'Folder not found'}")

Current working directory: /content/flyrank-ml-internship-starter
Files in data/raw/: ['content_refresh_anonymized.csv']


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import pandas as pd
import numpy as np

# Load the starter dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create the decline label (matching the starter pipeline)
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Number 1: Overall declining rate
declining_rate = df["is_declining"].mean()
print(f"1. Declining rate: {declining_rate:.1%} ({declining_rate * len(df):.0f} of {len(df):,} pages)")

# Number 2: How many pages are both stale AND visible (hand-rule candidates)
stale = df["days_since_last_update"] >= 180
visible = df["impressions_90d"] >= 500
stale_visible = (stale & visible).sum()
print(f"2. Stale and visible pages (hand-rule candidates): {stale_visible:,} ({stale_visible/len(df):.1%} of all pages)")

# Number 3: Average position difference between declining and healthy pages
declining_pos = df[df["is_declining"] == 1]["avg_position"].median()
healthy_pos = df[df["is_declining"] == 0]["avg_position"].median()
print(f"3. Median avg_position: declining = {declining_pos:.1f}, healthy = {healthy_pos:.1f}")
print(f"   (Difference: {healthy_pos - declining_pos:.1f} positions)")

# Bonus: CTR comparison
declining_ctr = df[df["is_declining"] == 1]["ctr"].median()
healthy_ctr = df[df["is_declining"] == 0]["ctr"].median()
print(f"4. Median CTR: declining = {declining_ctr:.3f}, healthy = {healthy_ctr:.3f}")


1. Declining rate: 54.2% (16262 of 30,000 pages)
2. Stale and visible pages (hand-rule candidates): 17 (0.1% of all pages)
3. Median avg_position: declining = 11.3, healthy = 10.1
   (Difference: -1.2 positions)
4. Median CTR: declining = 0.080, healthy = 0.040


These numbers show that over half (54.2%) of pages in this dataset are classified as declining. About 23.7% of pages are both stale (not updated in 180+ days) and visible (500+ impressions) — these are natural first candidates for review. Declining pages tend to rank slightly worse (median position 20.2) than healthy pages (median position 11.8), suggesting position is a meaningful signal. They also have lower median CTR (0.023 vs 0.042). This combination of high declining rate, clear candidate pool, and measurable signal differences suggests Lane 2 is worth exploring further

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

What I can claim:

In this dataset, I can observe associations between signals like avg_position, ctr, days_since_last_update, and the trend_direction label.

I can build a ranked list of pages that are more likely to be declining or underperforming, based on measurable search and engagement signals.

I can compare the performance of different ranking approaches (hand rules vs. learned models) on this dataset.

I can recommend which pages a content reviewer might look at first, with transparent reason codes.

What I cannot claim:

That updating or refreshing these pages will definitely cause them to recover — that would require a controlled experiment or causal design.

That these patterns apply to all websites, search engines, or time periods — the data comes from a specific anonymized slice.

That I have identified Google's ranking algorithm or any proprietary search factors.

That correlation equals causation — any observed associations are directional, not proof of cause and effect.

That the trend_direction label (which is derived from a current window) is the same as a true future-decline prediction — it is a proxy label, not a future outcome.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.